# ArXiv Pipeline Benchmarking 🚀

**Goal:** Measure the exact processing time (parsing + embedding) for a single arXiv PDF across different configurations of parsers (`PyPDF` vs `Docling`) and embedding models (`all-MiniLM-L6-v2` vs `BGE-M3`).

This notebook is entirely self-contained. You can run this end-to-end on Google Colab (using the free **T4 GPU**).

*Note: The first run of Docling will be slower because it downloads its vision models (approx 1GB).*


In [ ]:
# 1. Install Dependencies
!pip install -q pypdf docling sentence-transformers torch pandas matplotlib seaborn tqdm requests


In [ ]:
# 2. Imports and Setup
import os
import time
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
import torch

# Disable tokenizer parallelism warning
os.environ["TOKENIZERS_PARALLELISM"] = "false"

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print(f"✅ Using device: {device.upper()}")


In [ ]:
# 3. Download Sample PDFs
# Testing with just 1 famous paper (Attention is All You Need) for a rapid run. 
# You can uncomment the rest later for the full benchmark.
sample_arxiv_ids = [
    "1706.03762", 
    # "2005.11401", "2103.00020", "2303.08774", "2304.03277",
    # "2307.09288", "2310.06825", "2312.00752", "2401.04088", "2402.03300",
    # "2402.13753", "2403.05530", "2404.07143", "2405.04517", "2406.01258",
    # "2407.01392", "2408.02111", "2409.03456", "2410.01234", "2411.05678"
]

pdf_dir = "/tmp/benchmark_pdfs"
os.makedirs(pdf_dir, exist_ok=True)
pdf_paths = []

print(f"📥 Downloading {len(sample_arxiv_ids)} sample papers...")
for arxiv_id in tqdm(sample_arxiv_ids):
    pdf_path = os.path.join(pdf_dir, f"{arxiv_id}.pdf")
    pdf_paths.append(pdf_path)
    
    if not os.path.exists(pdf_path):
        url = f"https://export.arxiv.org/pdf/{arxiv_id}.pdf"
        headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}
        
        # Retry logic for rate limits
        for attempt in range(3):
            try:
                response = requests.get(url, headers=headers, stream=True, timeout=15)
                if response.status_code == 200:
                    with open(pdf_path, 'wb') as f:
                        for chunk in response.iter_content(chunk_size=8192):
                            if chunk:
                                f.write(chunk)
                    break # Success
                elif response.status_code == 429:
                    time.sleep(5) # Throttled, sleep longer
                else:
                    time.sleep(2)
            except Exception as e:
                time.sleep(2) # Connection broken, wait and retry
                
        time.sleep(1) # Polite delay between requests to avoid ban
            
# Filter only successfully downloaded (and completely downloaded) PDFs
valid_pdfs = []
for p in pdf_paths:
    if os.path.exists(p) and os.path.getsize(p) > 10000: # Ensure it's a real PDF, not a small error HTML
        valid_pdfs.append(p)
        
pdf_paths = valid_pdfs
print(f"✅ Successfully downloaded {len(pdf_paths)} valid PDFs.")


In [ ]:
# 4. Define Parsers
from pypdf import PdfReader
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.pipeline_options import PdfPipelineOptions, AcceleratorOptions, AcceleratorDevice
from docling.datamodel.base_models import InputFormat
from docling.chunking import HierarchicalChunker

# 4A. PyPDF (Sliding Window Chunker)
def parse_pypdf(pdf_path: str) -> list[str]:
    reader = PdfReader(pdf_path)
    full_text = []
    for page in reader.pages:
        text = page.extract_text()
        if text:
            full_text.append(text)
    
    text = " ".join(full_text)
    
    # Simple sliding window chunking
    chunk_size = 1500
    overlap = 200
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += (chunk_size - overlap)
    
    return chunks

# 4B. Docling (Layout Aware)
# Updated to use AUTO so it fully utilizes the Colab T4 GPU
pipeline_options = PdfPipelineOptions()
pipeline_options.accelerator_options = AcceleratorOptions(
    num_threads=4, device=AcceleratorDevice.AUTO 
)
_converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)
    }
)
_hierarchical_chunker = HierarchicalChunker()

def parse_docling(pdf_path: str) -> list[str]:
    dl_doc = _converter.convert(pdf_path).document
    chunk_iter = _hierarchical_chunker.chunk(dl_doc)
    
    chunks = []
    for docling_chunk in chunk_iter:
        text = docling_chunk.text
        if text and text.strip():
            chunks.append(text)
            
    return chunks

print("✅ Parsers defined.")


In [ ]:
# 5. Define Embedders
from sentence_transformers import SentenceTransformer

# Load models globally so we aren't reloading them every batch
print("Loading MiniLM...")
model_minilm = SentenceTransformer('all-MiniLM-L6-v2', device=device)

print("Loading BGE-M3...")
model_bgem3 = SentenceTransformer('BAAI/bge-m3', device=device)

def embed_minilm(chunks: list[str]) -> np.ndarray:
    if not chunks:
        return np.array([])
    # Maximized for T4 GPU
    return model_minilm.encode(chunks, batch_size=512, show_progress_bar=False)

def embed_bgem3(chunks: list[str]) -> np.ndarray:
    if not chunks:
        return np.array([])
    # Maximized for T4 GPU (16GB VRAM)
    return model_bgem3.encode(chunks, batch_size=128, show_progress_bar=False)

print("✅ Embedders loaded and defined.")


In [ ]:
# 6. Benchmark Runner
# Warmup Docling because it downloads models on the very first parse
print("Warming up Docling...")
if len(pdf_paths) > 0:
    _ = parse_docling(pdf_paths[0])
print("✅ Warmup complete.")

configurations = [
    {"name": "PyPDF + MiniLM", "parser": parse_pypdf, "embedder": embed_minilm},
    {"name": "PyPDF + BGE-M3", "parser": parse_pypdf, "embedder": embed_bgem3},
    {"name": "Docling + MiniLM", "parser": parse_docling, "embedder": embed_minilm},
    {"name": "Docling + BGE-M3", "parser": parse_docling, "embedder": embed_bgem3},
]

results = []

for config in configurations:
    print(f"\n🚀 Running Benchmark: {config['name']}")
    
    total_parse_time = 0
    total_embed_time = 0
    total_chunks = 0
    successful_pdfs = 0
    
    for pdf in tqdm(pdf_paths, desc=f"{config['name']}"):
        try:
            # Time Parse
            t0 = time.perf_counter()
            chunks = config["parser"](pdf)
            t1 = time.perf_counter()
            
            # Time Embed
            t2 = time.perf_counter()
            _ = config["embedder"](chunks)
            t3 = time.perf_counter()
            
            total_parse_time += (t1 - t0)
            total_embed_time += (t3 - t2)
            total_chunks += len(chunks)
            successful_pdfs += 1
        except Exception as e:
            print(f"Error processing {pdf} with {config['name']}: {e}")
            
    if successful_pdfs > 0:
        avg_parse = total_parse_time / successful_pdfs
        avg_embed = total_embed_time / successful_pdfs
        avg_total = avg_parse + avg_embed
        
        # Extrapolate for 150k papers (hours)
        est_150k_hours = (avg_total * 150_000) / 3600
        
        results.append({
            "Configuration": config['name'],
            "Avg Chunks / PDF": round(total_chunks / successful_pdfs),
            "Parse (s/PDF)": round(avg_parse, 3),
            "Embed (s/PDF)": round(avg_embed, 3),
            "Total (s/PDF)": round(avg_total, 3),
            "Est. 150k Papers": f"{round(est_150k_hours, 1)} Hours"
        })

df_results = pd.DataFrame(results)

# Export results to CSV so they can be easily downloaded from Colab
df_results.to_csv('benchmark_results.csv', index=False)
print("\n✅ Benchmark Complete! Results saved to 'benchmark_results.csv'")


In [ ]:
# 7. Results & Visualization
from IPython.display import display

# Print Table
print("📊 BENCHMARK RESULTS (Averages per PDF)\n")
display(df_results)

# Plotting the data beautifully
sns.set_theme(style="whitegrid")
plt.figure(figsize=(10, 6))

# Prepare data for stacked bar chart
config_names = df_results['Configuration']
parse_times = df_results['Parse (s/PDF)']
embed_times = df_results['Embed (s/PDF)']

# Bar positions
x = np.arange(len(config_names))
width = 0.6

# Create stacked bars
plt.bar(x, parse_times, width, label='Parsing Time', color='#4C72B0')
plt.bar(x, embed_times, width, bottom=parse_times, label='Embedding Time', color='#55A868')

# Formatting
plt.title('Average Processing Time per PDF by Configuration', fontsize=14, pad=15)
plt.ylabel('Time in Seconds (Log Scale for readability)', fontsize=12)
plt.yscale('log') # Log scale because docling will dwarf PyPDF
plt.xticks(x, config_names, rotation=15, ha="right", fontsize=11)
plt.legend(title='Process Stage')

# Annotate Total times on top of bars
for i, total in enumerate(df_results['Total (s/PDF)']):
    plt.text(i, total + (total * 0.1), f"{total}s", ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()
